# 03 — Model Training and Evaluation

This notebook trains FXGuard risk classification models.

Models compared:
- Logistic Regression
- Random Forest
- simple baselines

The project script `scripts/train_models.py` also trains and saves the best final models used by the backend.

In [9]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

ROOT = Path.cwd().parent
DATA_DIR = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    'mid_rate', 'daily_return', 'return_7d', 'return_14d', 'ma_7', 'ma_14', 'ma_30',
    'ma_gap', 'volatility_7d', 'volatility_14d', 'volatility_30d', 'momentum_7d',
    'momentum_14d', 'spread', 'spread_pct', 'depreciation_days_7d', 'depreciation_days_14d'
]
CLASS_ORDER = ['Low', 'Medium', 'High']
CLASS_TO_INT = {label: idx for idx, label in enumerate(CLASS_ORDER)}
INT_TO_CLASS = {idx: label for label, idx in CLASS_TO_INT.items()}


In [10]:
def load_model_ready(horizon):
    path = DATA_DIR / f'exchange_rates_model_ready_{horizon}d_4year.csv'
    label = f'risk_label_{horizon}d'
    df = pd.read_csv(path, parse_dates=['date']).sort_values('date').reset_index(drop=True)
    df = df.dropna(subset=FEATURE_COLUMNS + [label]).copy()
    return df, label

def chronological_split(df, train_ratio=0.8):
    split_idx = int(len(df) * train_ratio)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def evaluate_model(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

In [11]:
def train_horizon(horizon):
    df, label = load_model_ready(horizon)
    train_df, test_df = chronological_split(df, 0.8)

    X_train, y_train = train_df[FEATURE_COLUMNS], train_df[label]
    X_test, y_test = test_df[FEATURE_COLUMNS], test_df[label]
    y_train_xgb = y_train.map(CLASS_TO_INT)

    # Baseline 1: always predict the most common class in training set
    majority_class = y_train.mode()[0]
    baseline_majority = np.array([majority_class] * len(y_test))

    # Baseline 2: volatility threshold rule
    # This is intentionally simple for comparison.
    vol_rule = []
    q50 = train_df['volatility_7d'].quantile(0.50)
    q80 = train_df['volatility_7d'].quantile(0.80)
    for v in test_df['volatility_7d']:
        if v <= q50:
            vol_rule.append('Low')
        elif v <= q80:
            vol_rule.append('Medium')
        else:
            vol_rule.append('High')

    models = {
        'majority_baseline': baseline_majority,
        'volatility_threshold_baseline': np.array(vol_rule),
        'logistic_regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42))
        ]),
        'random_forest': RandomForestClassifier(
            n_estimators=350,
            max_depth=9,
            min_samples_leaf=4,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        ),
        'xgboost': XGBClassifier(
            objective='multi:softprob',
            num_class=len(CLASS_ORDER),
            n_estimators=250,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=42,
            tree_method='hist',
            eval_metric='mlogloss'
        )
    }

    results = {}
    fitted_models = {}
    for name, model in models.items():
        if isinstance(model, np.ndarray):
            preds = model
        elif name == 'xgboost':
            model.fit(X_train, y_train_xgb)
            preds = model.predict(X_test)
            preds = pd.Series(preds).map(INT_TO_CLASS).to_numpy()
            fitted_models[name] = model
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            fitted_models[name] = model
        results[name] = evaluate_model(y_test, preds)

    results_df = pd.DataFrame(results).T.sort_values('f1_macro', ascending=False)
    best_name = results_df.index[0]
    best_model = fitted_models.get(best_name)
    if best_model is not None:
        joblib.dump(best_model, MODEL_DIR / f'risk_model_{horizon}d.pkl')

    print(f'{horizon}-day model results')
    display(results_df)
    print('Best model:', best_name)
    if best_model is not None:
        preds = best_model.predict(X_test)
        print(classification_report(y_test, preds, zero_division=0))
        print('Confusion matrix labels:', CLASS_ORDER)
        print(confusion_matrix(y_test, preds, labels=CLASS_ORDER))
    return results_df


In [12]:
results_7d = train_horizon(7)

7-day model results


,accuracy,precision_macro,recall_macro,f1_macro
volatility_threshold_baseline,1.0,1.0,1.0,1.0
logistic_regression,1.0,1.0,1.0,1.0
random_forest,1.0,1.0,1.0,1.0
xgboost,1.0,1.0,1.0,1.0
majority_baseline,0.0,0.0,0.0,0.0


Best model: volatility_threshold_baseline


In [13]:
results_14d = train_horizon(14)

14-day model results


,accuracy,precision_macro,recall_macro,f1_macro
volatility_threshold_baseline,1.0,1.0,1.0,1.0
logistic_regression,1.0,1.0,1.0,1.0
random_forest,1.0,1.0,1.0,1.0
xgboost,1.0,1.0,1.0,1.0
majority_baseline,0.0,0.0,0.0,0.0


Best model: volatility_threshold_baseline


## Train using the project script

The command below runs the official training script used by the backend. It saves:

- `backend/models/risk_model_7d.pkl`
- `backend/models/risk_model_14d.pkl`
- `backend/models/model_metadata.json`

In [ ]:
# Retrain final backend models from inside the notebook.
# running: python scripts/train_models.py

import subprocess, sys
result = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'train_models.py')], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Training 7-day model...
Training 14-day model...
Saved metadata: C:\Users\Prime core\Downloads\FXGuard AI\FXGuard_AI_Project\backend\models\model_metadata.json
Done.




In [1]:
from pathlib import Path
import json

# Ensure ROOT and MODEL_DIR exist when running this cell standalone
try:
    ROOT
except NameError:
    ROOT = Path.cwd().parent

try:
    MODEL_DIR
except NameError:
    MODEL_DIR = ROOT / 'backend' / 'models'
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = MODEL_DIR / 'model_metadata.json'
if metadata_path.exists():
    try:
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        print('Metadata keys:', list(metadata.keys()))
        print('Models in metadata:', list(metadata.get('models', {}).keys()))
    except Exception as e:
        print('Failed to read model metadata:', e)
else:
    print('No metadata found yet. Run the training script first.')


Metadata keys: ['project', 'currency_pair', 'models']
Models in metadata: ['7d', '14d']
